# s01 — The Loop

**What this teaches:** the fundamental shape of every Omnis agent — a `model → tool → model` cycle managed by ADK. With no tools attached, the loop terminates after a single model turn. Once you understand this, every later example just adds capabilities **into** this loop without changing it.

**Concept anchor:** an "agent" in Omnis is not magic — it is *a model* + *a runner* that calls the model in a loop until no tool calls are emitted. Everything else (tools, plugins, sub-agents, compression, permissions) is a participant inside that loop.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).

## 1. Imports

Two packages do almost all the work in this example: `agentkit` wires the agent and runner; `stream` prints the streamed model output to stdout.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Construct the model

`agentkit.NewModel` reads `OMNIS_PROVIDER`, the matching API key env var, and any overrides in `config/agent.yaml`. It returns an ADK `model.Model` ready to be plugged into an agent. The value of `llm` is reused by every later cell — even though we re-create it here for visibility, you only need this once per session.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
fmt.Println("model wired:", llm)

## 4. Build the agent

An agent in this example is intentionally minimal — no tools, no instructions, no sub-agents. The `Name` and `Description` are only used for tracing/logging.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s01_loop",
    Description: "Bare agent loop demo.",
    Model:       llm,
})
must(err)
fmt.Println("agent built:", a.Name())

## 5. Build the runner

The **runner** owns the loop. Calling `RunOnce(ctx, r, prompt)` walks the model→tool→model cycle to completion and returns an event stream. Plugins (event bus, permissions, compression) are attached here as variadic arguments — none in this minimal example.

In [ ]:
r, err := agentkit.Runner("s01", a)
must(err)

## 6. Run the loop

Because we attached zero tools, the model will respond once and stop. There are no tool calls to dispatch, so the loop terminates after the very first turn. Watch the stream as it arrives.

In [ ]:
prompt := "Say hello and tell me what 'the agent loop' means in one sentence."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The output is a single model response with no tool-call detours. Compare this with [s05_tools](../s05_tools/s05_tools.ipynb), where the same loop now dispatches multiple `bash`/`read`/`write` calls inside one `RunOnce`.
- The runner's first argument (`"s01"`) is the *session ID*. Re-running the cell reuses the conversation — try it after `s16_resume`.

## Try it yourself

1. Change the prompt to ask a multi-step question (e.g. "plan a trip to Mars in 5 steps") and observe that the loop still terminates after one turn — without tools, the model can't *do* anything, it can only describe.
2. Set `OMNIS_DEBUG=1` before launching Jupyter and re-run: the runner prints every event the loop emits.